# Base de Dados Simulada — Opinate | Tema: Mudanças na CNH no Brasil

**Insumos:** `database_opinate.pdf` (modelagem ER) + `raspagem_CNH.ipynb` (scraper de comentários do YouTube)

## Sobre a procedência dos dados

Este notebook gera uma base de dados **simulada porém estruturalmente fiel** ao
schema do Opinate, populada com o tema real do projeto: *mudanças no processo
e regras da CNH no Brasil*.

Duas observações importantes sobre os insumos, por transparência:

1. **`raspagem_CNH.ipynb` contém apenas o código do scraper, não dados já
   coletados.** O notebook define funções que consultam a YouTube Data API
   v3 (`buscar_videos`, `buscar_comentarios`, `limpar_texto` etc.), mas não
   há, nos anexos, nenhum arquivo de saída (`comentarios_cnh.csv`) com
   comentários reais já raspados — e este ambiente não tem acesso à API do
   YouTube para executá-lo agora. Por isso, o pipeline abaixo foi desenhado
   em **duas camadas**: (a) se o arquivo `comentarios_cnh.csv` existir no
   diretório de trabalho — produto real do scraper —, ele é carregado e
   usado; (b) caso contrário, um corpus de comentários **sintético, porém
   plausível e 100% autoral**, no mesmo formato de saída do scraper original,
   é usado no lugar. Isso significa que, no dia em que você rodar o scraper
   de verdade, basta colocar o CSV gerado na mesma pasta e reexecutar este
   notebook — nenhuma outra mudança é necessária.
2. **Alerta de segurança:** o notebook de raspagem fornecido contém uma
   `API_KEY` da Google/YouTube **diretamente no código-fonte**. Como esse
   arquivo já foi compartilhado, recomendamos **revogar essa chave no Google
   Cloud Console e gerar uma nova**, passando a carregá-la por variável de
   ambiente (`os.environ["YOUTUBE_API_KEY"]`) em vez de hardcoded. Chaves
   versionadas/compartilhadas em texto puro são um risco de segurança comum
   e fácil de evitar.

## Estrutura deste notebook

| Passo | Conteúdo | Tabelas geradas |
|---|---|---|
| 0 | Configuração e seed | — |
| 4 | Geografia (gerado primeiro por dependência de FK) | `region`, `state`, `municipality` |
| 1 | Perfil de usuário | `app_user` |
| 2 | Propostas com tema da CNH | `proposal` |
| 3 | Comentários e reações (lógica da raspagem aplicada aqui) | `comment`, `comment_vote`, `proposal_vote`, `proposal_support`, `proposal_share` |
| 5 | Validação / smoke tests | testes de integridade referencial e tipos |
| 6 | Persistência (bônus) | banco SQLite com FKs reais, incluindo teste negativo de violação |


## Passo 0 — Imports, seed e configuração geral

In [2]:
%pip install pandas numpy matplotlib faker 

Note: you may need to restart the kernel to use updated packages.


In [3]:
import random
import unicodedata
from datetime import datetime, timedelta

import numpy as np
import pandas as pd


from faker import Faker

# ---------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)    
fake = Faker("pt_BR")
Faker.seed(SEED)

N_USERS = 600
NOW = datetime(2026, 6, 19, 12, 0, 0)


def random_datetime(days_back_min, days_back_max):
    delta_days = random.uniform(days_back_min, days_back_max)
    return NOW - timedelta(days=delta_days)

## Passo 4 — Localidades (`region`, `state`, `municipality`)

Geradas primeiro porque `app_user.municipality_id` depende de `municipality_id`,
que por sua vez depende de `state_id` e `region_id`. Usamos as 5 regiões e os
27 estados/DF reais do Brasil, com as respectivas capitais como municípios-base,
mais algumas cidades adicionais de grande porte para variar a distribuição
geográfica dos usuários.

In [3]:
REGIOES = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]

ESTADOS = [
    ("Acre", "Norte"), ("Amapá", "Norte"), ("Amazonas", "Norte"),
    ("Pará", "Norte"), ("Rondônia", "Norte"), ("Roraima", "Norte"), ("Tocantins", "Norte"),
    ("Alagoas", "Nordeste"), ("Bahia", "Nordeste"), ("Ceará", "Nordeste"),
    ("Maranhão", "Nordeste"), ("Paraíba", "Nordeste"), ("Pernambuco", "Nordeste"),
    ("Piauí", "Nordeste"), ("Rio Grande do Norte", "Nordeste"), ("Sergipe", "Nordeste"),
    ("Distrito Federal", "Centro-Oeste"), ("Goiás", "Centro-Oeste"),
    ("Mato Grosso", "Centro-Oeste"), ("Mato Grosso do Sul", "Centro-Oeste"),
    ("Espírito Santo", "Sudeste"), ("Minas Gerais", "Sudeste"),
    ("Rio de Janeiro", "Sudeste"), ("São Paulo", "Sudeste"),
    ("Paraná", "Sul"), ("Rio Grande do Sul", "Sul"), ("Santa Catarina", "Sul"),
]

CAPITAIS = {
    "Acre": "Rio Branco", "Amapá": "Macapá", "Amazonas": "Manaus", "Pará": "Belém",
    "Rondônia": "Porto Velho", "Roraima": "Boa Vista", "Tocantins": "Palmas",
    "Alagoas": "Maceió", "Bahia": "Salvador", "Ceará": "Fortaleza", "Maranhão": "São Luís",
    "Paraíba": "João Pessoa", "Pernambuco": "Recife", "Piauí": "Teresina",
    "Rio Grande do Norte": "Natal", "Sergipe": "Aracaju", "Distrito Federal": "Brasília",
    "Goiás": "Goiânia", "Mato Grosso": "Cuiabá", "Mato Grosso do Sul": "Campo Grande",
    "Espírito Santo": "Vitória", "Minas Gerais": "Belo Horizonte",
    "Rio de Janeiro": "Rio de Janeiro", "São Paulo": "São Paulo",
    "Paraná": "Curitiba", "Rio Grande do Sul": "Porto Alegre", "Santa Catarina": "Florianópolis",
}

CIDADES_EXTRA = {
    "São Paulo": ["Campinas", "Guarulhos", "Sorocaba", "Santos"],
    "Rio de Janeiro": ["Niterói", "Duque de Caxias"],
    "Minas Gerais": ["Uberlândia", "Juiz de Fora"],
    "Paraná": ["Londrina", "Cascavel"],
    "Rio Grande do Sul": ["Caxias do Sul", "Pelotas"],
    "Santa Catarina": ["Joinville", "Blumenau"],
    "Bahia": ["Feira de Santana"],
    "Pernambuco": ["Caruaru"],
    "Ceará": ["Juazeiro do Norte"],
}

df_region = pd.DataFrame({
    "region_id": range(1, len(REGIOES) + 1),
    "name": REGIOES,
})
region_id_map = dict(zip(df_region["name"], df_region["region_id"]))

df_state = pd.DataFrame({
    "state_id": range(1, len(ESTADOS) + 1),
    "name": [e[0] for e in ESTADOS],
    "region_id": [region_id_map[e[1]] for e in ESTADOS],
})
state_id_map = dict(zip(df_state["name"], df_state["state_id"]))

municipios_rows = []
for estado, capital in CAPITAIS.items():
    municipios_rows.append((capital, estado))
for estado, cidades in CIDADES_EXTRA.items():
    for c in cidades:
        municipios_rows.append((c, estado))

df_municipality = pd.DataFrame({
    "municipality_id": range(1, len(municipios_rows) + 1),
    "name": [m[0] for m in municipios_rows],
    "state_id": [state_id_map[m[1]] for m in municipios_rows],
})

print("region:", df_state.shape, df_municipality.shape)
print(df_region.head())
print(df_state.head())
print(df_municipality.head())

region: (27, 3) (44, 3)
   region_id          name
0          1         Norte
1          2      Nordeste
2          3  Centro-Oeste
3          4       Sudeste
4          5           Sul
   state_id      name  region_id
0         1      Acre          1
1         2     Amapá          1
2         3  Amazonas          1
3         4      Pará          1
4         5  Rondônia          1
   municipality_id         name  state_id
0                1   Rio Branco         1
1                2       Macapá         2
2                3       Manaus         3
3                4        Belém         4
4                5  Porto Velho         5


## Passo 1 — Perfil do usuário (`app_user`)

Geração de usuários sintéticos com `Faker` (locale `pt_BR`) para nomes e domínios
de e-mail realistas. A **idade** segue uma distribuição normal (média 34, desvio
13) truncada entre 16 e 85 anos — coerente com a faixa etária que de fato
participa de plataformas de participação cidadã. O **gênero** segue uma
distribuição com leve predominância de masculino/feminino e uma fração minoritária
de não-binário / prefere não informar, refletindo pesquisas de autodeclaração de
gênero em plataformas digitais brasileiras.

In [4]:
def slugify(name):
    nfkd = unicodedata.normalize("NFKD", name)
    ascii_name = nfkd.encode("ascii", "ignore").decode("ascii")
    return ascii_name.lower().replace(" ", ".")


GENEROS = ["Feminino", "Masculino", "Não-binário", "Prefiro não informar"]
GENERO_PROB = [0.47, 0.47, 0.04, 0.02]

municipality_ids = df_municipality["municipality_id"].tolist()

emails_usados = set()
user_rows = []
for uid in range(1, N_USERS + 1):
    nome = fake.name()
    base_email = slugify(nome)
    email = f"{base_email}@{fake.free_email_domain()}"
    sufixo = 1
    while email in emails_usados:
        sufixo += 1
        email = f"{base_email}{sufixo}@{fake.free_email_domain()}"
    emails_usados.add(email)

    idade = int(np.clip(np.random.normal(loc=34, scale=13), 16, 85))
    genero = np.random.choice(GENEROS, p=GENERO_PROB)
    municipio_id = random.choice(municipality_ids)
    criado_em = random_datetime(30, 730)

    user_rows.append({
        "user_id": uid,
        "name": nome,
        "email": email,
        "age": idade,
        "gender": genero,
        "municipality_id": municipio_id,
        "created_at": criado_em,
    })

df_app_user = pd.DataFrame(user_rows)
print("\napp_user:", df_app_user.shape)
print(df_app_user.head())
print(df_app_user["age"].describe())
print(df_app_user["gender"].value_counts(normalize=True))


app_user: (600, 7)
   user_id                    name                               email  age  \
0        1            Brenda Alves             brenda.alves@bol.com.br   40   
1        2  Henry Gabriel Carvalho  henry.gabriel.carvalho@hotmail.com   32   
2        3             Ísis Borges          isis.borges@outlook.com.br   30   
3        4           Arthur Borges            arthur.borges@uol.com.br   30   
4        5          Thiago Pereira            thiago.pereira@gmail.com   54   

      gender  municipality_id                 created_at  
0  Masculino               41 2026-03-03 13:38:16.997340  
1  Masculino               18 2025-11-30 01:49:00.681966  
2   Feminino                9 2024-12-20 23:17:00.967361  
3  Masculino               44 2024-12-18 00:46:54.802499  
4   Feminino               35 2026-03-20 15:25:39.402573  
count    600.000000
mean      34.403333
std       11.681528
min       16.000000
25%       26.000000
50%       34.000000
75%       42.000000
max       8

## Passo 2 — Carga temática da CNH (`proposal`)

Em vez de inventar uma proposta genérica, usamos diretamente os termos de busca
(`TERMOS_BUSCA`) definidos em `raspagem_CNH.ipynb` — cada termo de pesquisa do
scraper (ex.: *"novas regras CNH 2024 2025"*, *"CNH autoescola obrigatório
mudança lei"*, *"reforma CNH DENATRAN"*) vira o tema de uma proposta de discussão
pública, com título e descrição originais escritos para refletir o assunto real
pesquisado. Isso preserva a rastreabilidade entre o insumo de raspagem e a base
final.

In [5]:
import random
import pandas as pd

# A base real (dataset_final_cnh.csv) tem um único valor na coluna 'tema':
# "Novas regras CNH" -> 727 comentários. Por isso, diferente da versão
# anterior (que previa 6 propostas, uma por termo de busca), aqui geramos
# UMA ÚNICA proposta de discussão pública, que concentra todos os
# comentários raspados sobre o tema.
#
# IMPORTANTE: "termo_origem" precisa bater exatamente com o valor da coluna
# 'tema' do CSV ("Novas regras CNH"), para que o resolver_proposal_id()
# do script de comentários encontre o match direto, sem cair no fallback
# aleatório.
PROPOSTAS_CNH = [
    {
        "termo_origem": "Novas regras CNH",
        "title": "Novas regras CNH",
        "description": (
            "Discussão sobre o conjunto de mudanças recentes no processo de "
            "emissão da Carteira Nacional de Habilitação, incluindo prazos "
            "de validade, exigência (ou não) de curso teórico em autoescola, "
            "simplificação das etapas até a emissão do documento, "
            "modernização e digitalização dos serviços do DETRAN, ampliação "
            "do acesso via CNH Social e revisão das regras do exame "
            "toxicológico para condutores."
        ),
    },
]

proposal_rows = []
for i, tema in enumerate(PROPOSTAS_CNH, start=1):
    proposal_rows.append({
        "proposal_id": i,
        "title": tema["title"],
        "description": tema["description"],
        "created_by": random.choice(df_app_user["user_id"].tolist()),
        "created_at": random_datetime(60, 540),
        "_termo_origem": tema["termo_origem"],  # auxiliar, removido depois
    })

df_proposal_full = pd.DataFrame(proposal_rows)
df_proposal = df_proposal_full.drop(columns=["_termo_origem"])
termo_to_proposal_id = dict(zip(df_proposal_full["_termo_origem"], df_proposal_full["proposal_id"]))

print("\nproposal:", df_proposal.shape)
print(df_proposal.head(10))


proposal: (1, 5)
   proposal_id             title  \
0            1  Novas regras CNH   

                                         description  created_by  \
0  Discussão sobre o conjunto de mudanças recente...         244   

                  created_at  
0 2025-07-11 20:24:51.616724  


## Passo 3 — Comentários e reações (`comment`, `comment_vote`, `proposal_vote`, `proposal_support`, `proposal_share`)

**Onde a lógica da raspagem é reaplicada:**

- A função `limpar_texto()` é **copiada literalmente** de `raspagem_CNH.ipynb`
  (mesma regex de limpeza de emojis/caracteres de controle), e usada para
  popular `comment.content` — exatamente como o scraper faria com o
  `comentario_limpo` real.
- O pipeline primeiro tenta ler `comentarios_cnh.csv` (saída real do scraper).
  Na ausência dele neste ambiente, usamos um corpus sintético organizado pelos
  mesmos 6 temas das propostas, no mesmo formato de colunas do scraper
  (`comentario`, `likes_comentario`, `respostas`, `data_comentario`).
- A métrica `likes_comentario` — que no scraper real vem da API do YouTube —
  é **mapeada para linhas individuais de `comment_vote`**: cada like vira uma
  linha `vote_type='like'` associada a um usuário diferente do autor, e uma
  fração menor de discordância gera `vote_type='dislike'`, simulando o padrão
  real de engajamento (poucos comentários "virais", maioria com poucas
  reações).

In [7]:
import os
import glob
import random
import re
import pandas as pd

# Caminho para o CSV real
CSV_REAL = "../Algoritmo Raspagem de dados/Dados raspados/dataset_opinioes_cnh_youtube.csv"


def localizar_csv(caminho_preferido):
    """Tenta achar o CSV mesmo se o caminho relativo não bater com o cwd atual.
    Ordem de tentativa: (1) caminho exatamente como informado, (2) caminho
    absoluto a partir do cwd, (3) busca por nome de arquivo a partir do cwd
    e de /content (padrão Colab)."""
    candidatos = [
        caminho_preferido,
        os.path.abspath(caminho_preferido),
    ]

    nome_arquivo = os.path.basename(caminho_preferido)
    for raiz in (os.getcwd(), "/content", "/content/drive/MyDrive"):
        candidatos.extend(glob.glob(os.path.join(raiz, "**", nome_arquivo), recursive=True))

    for c in candidatos:
        if c and os.path.exists(c):
            return c
    return None


def limpar_texto(texto):
    """Função IDÊNTICA à de raspagem_CNH.ipynb: remove emojis/controle, mantém acentuação."""
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r"[\x00-\x1f\x7f]", " ", texto)
    texto = re.sub(r"[^\x00-\x7Fà-üÀ-Ü0-9\s\.,;:!?\"'()\-]", "", texto)
    return re.sub(r"\s+", " ", texto).strip()


def normalizar(s):
    """Normaliza string para comparação tolerante (minúsculas, sem espaços extras)."""
    return re.sub(r"\s+", " ", str(s).strip().lower())


# --- POPULANDO O CORPUS_POR_TEMA DINAMICAMENTE A PARTIR DO CSV ---
CORPUS_POR_TEMA = {}
LIKES_POR_TEMA = {}  # guarda os likes reais alinhados ao mesmo índice dos comentários

# DIAGNÓSTICO: mostra exatamente onde o Python está procurando o arquivo,
# antes de qualquer try/except engolir o erro.
print("Diretório de trabalho atual (cwd):", os.getcwd())
print("Caminho informado (CSV_REAL):", CSV_REAL)
print("Caminho absoluto resolvido a partir do cwd:", os.path.abspath(CSV_REAL))

caminho_encontrado = localizar_csv(CSV_REAL)
print("Arquivo encontrado em:", caminho_encontrado)

try:
    if caminho_encontrado is None:
        raise FileNotFoundError(
            f"Não encontrei '{CSV_REAL}' nem em {os.getcwd()}, /content ou "
            f"/content/drive/MyDrive. Confirme o caminho exato do arquivo "
            f"(use !find / -name 'dataset_final_cnh*.csv' no Colab para localizar)."
        )

    print(f"Carregando e mapeando dados reais de {caminho_encontrado} para o CORPUS_POR_TEMA...")
    df_csv_base = pd.read_csv(caminho_encontrado)
    print("Linhas carregadas:", len(df_csv_base), "| Colunas:", df_csv_base.columns.tolist())

    # MUDANÇA 1: a base real usa a coluna 'tema', não 'termo_busca'.
    # (sua base só tem 1 valor único: "Novas regras CNH" -> 727 comentários)
    coluna_tema = "tema" if "tema" in df_csv_base.columns else "termo_busca"

    CORPUS_POR_TEMA = (
        df_csv_base.groupby(coluna_tema)["comentario"]
        .apply(list)
        .to_dict()
    )

    # MUDANÇA 2: aproveitamos os likes REAIS da coluna 'likes' (em vez de só simular),
    # alinhados na mesma ordem que os comentários de cada tema.
    if "likes" in df_csv_base.columns:
        LIKES_POR_TEMA = (
            df_csv_base.groupby(coluna_tema)["likes"]
            .apply(list)
            .to_dict()
        )
except Exception as e:
    # Agora o erro REAL aparece (antes "FileNotFoundError, Exception" escondia tudo
    # atrás da mesma mensagem genérica, mascarando a causa verdadeira).
    print(f"Aviso: Não foi possível ler o CSV ({type(e).__name__}: {e}). Usando fallback sintético antigo.")
    CORPUS_POR_TEMA = {
        "novas regras CNH 2024 2025": [
            "Finalmente uma mudança que faz sentido, o processo estava muito travado.",
            "Não entendi direito as novas regras, alguém pode explicar melhor?",
        ],
        "CNH autoescola obrigatório mudança lei": [
            "Sou instrutor de autoescola e acho que isso vai prejudicar a segurança viária.",
            "Já era hora de poder estudar sozinho, gastei uma fortuna com aula teórica.",
        ],
        "processo CNH simplificado Brasil": [
            "Demorei quase um ano pra tirar minha CNH, simplificar é urgente.",
        ],
        "reforma CNH DENATRAN": [
            "A CNH digital já ajudou bastante, mas o atendimento presencial ainda é lento.",
        ],
        "mudança processo tirar CNH Brasil": [
            "CNH social mudou minha vida, consegui emprego de motorista depois.",
        ],
        "nova lei CNH habilitação Brasil": [
            "O exame toxicológico é caro demais para quem é categoria B também.",
        ],
    }


def gerar_likes_realistas():
    """Distribuição assimétrica: maioria com poucos likes, raros 'virais'.
    Usado apenas como fallback quando não há likes reais disponíveis."""
    r = random.random()
    if r < 0.70:
        return random.randint(0, 8)
    elif r < 0.93:
        return random.randint(9, 60)
    elif r < 0.99:
        return random.randint(61, 250)
    else:
        return random.randint(251, 600)


# MUDANÇA 3: mapeamento tolerante de tema -> proposal_id.
# Tenta achar a chave exata em termo_to_proposal_id; se não achar, tenta
# uma correspondência normalizada (case-insensitive); só então cai no random.
def resolver_proposal_id(termo, termo_to_proposal_id, available_proposal_ids):
    if termo in termo_to_proposal_id:
        return termo_to_proposal_id[termo]

    termo_norm = normalizar(termo)
    for chave, pid in termo_to_proposal_id.items():
        if normalizar(chave) == termo_norm or termo_norm in normalizar(chave) or normalizar(chave) in termo_norm:
            return pid

    # Nenhuma correspondência: avisa e usa fallback aleatório
    print(f"Aviso: tema '{termo}' não encontrado em termo_to_proposal_id. Usando proposta aleatória.")
    return random.choice(available_proposal_ids)


comment_rows = []
comment_id_counter = 1
all_user_ids = df_app_user["user_id"].tolist()
available_proposal_ids = df_proposal["proposal_id"].tolist()

try:
    if not CORPUS_POR_TEMA:
        raise ValueError("Corpus está vazio.")

    print("Processando e gerando linhas do dataframe de comentários final...")
    for termo, lista_comentarios in CORPUS_POR_TEMA.items():
        proposal_id = resolver_proposal_id(termo, termo_to_proposal_id, available_proposal_ids)
        likes_reais = LIKES_POR_TEMA.get(termo, [])

        for idx, texto in enumerate(lista_comentarios):
            autor_id = random.choice(all_user_ids)

            # MUDANÇA 4: usa o like real da linha correspondente se existir,
            # senão cai no gerador sintético (mantém o pipeline funcionando
            # mesmo no fallback estático).
            if idx < len(likes_reais):
                likes_comentario = int(likes_reais[idx])
            else:
                likes_comentario = gerar_likes_realistas()

            comment_rows.append({
                "comment_id": comment_id_counter,
                "proposal_id": proposal_id,
                "user_id": autor_id,
                "content": limpar_texto(texto),
                "created_at": random_datetime(1, 480),
                "aux_likes_comentario": likes_comentario,
                "aux_respostas": random.randint(0, 12),
            })
            comment_id_counter += 1

except Exception as e:
    print(f"Erro ao processar o pipeline final: {e}")

df_comment_full = pd.DataFrame(comment_rows)
df_comment = df_comment_full.drop(columns=["aux_likes_comentario", "aux_respostas"])

Diretório de trabalho atual (cwd): c:\Users\juanc\OneDrive\Documentos\Arquivos\Arquivos Juan\FATEC\Fatec 2026\Projeto Integrador III\PI III\Algoritmo Análise de sentimentos
Caminho informado (CSV_REAL): ../Algoritmo Raspagem de dados/Dados raspados/dataset_opinioes_cnh_youtube.csv
Caminho absoluto resolvido a partir do cwd: c:\Users\juanc\OneDrive\Documentos\Arquivos\Arquivos Juan\FATEC\Fatec 2026\Projeto Integrador III\PI III\Algoritmo Raspagem de dados\Dados raspados\dataset_opinioes_cnh_youtube.csv
Arquivo encontrado em: ../Algoritmo Raspagem de dados/Dados raspados/dataset_opinioes_cnh_youtube.csv
Carregando e mapeando dados reais de ../Algoritmo Raspagem de dados/Dados raspados/dataset_opinioes_cnh_youtube.csv para o CORPUS_POR_TEMA...
Linhas carregadas: 927 | Colunas: ['tema', 'video_id', 'comentario', 'likes', 'autor', 'fonte']
Processando e gerando linhas do dataframe de comentários final...


Com os comentários e seus `likes_comentario` em mãos, geramos as tabelas de votos/engajamento restantes do schema (`comment_vote`, `proposal_vote`, `proposal_support`, `proposal_share`), sempre amostrando usuários reais da tabela `app_user` para preservar a integridade referencial.

In [8]:
# --- comment_vote: mapeia likes_comentario -> linhas individuais de voto ----
# Cada like real vira uma linha (comment_id, user_id, 'like'); uma fração
# menor de "dislikes" também é simulada para dar realismo (todo vídeo tem
# gente discordando no comentário mais curtido).
all_user_ids = df_app_user["user_id"].tolist()
comment_vote_rows = []
comment_vote_id = 1

for row in df_comment_full.itertuples():
    autor = row.user_id
    likes = row.aux_likes_comentario
    candidatos = [u for u in all_user_ids if u != autor]

    n_likes = min(likes, len(candidatos))
    votantes_like = random.sample(candidatos, n_likes)

    restantes = [u for u in candidatos if u not in votantes_like]
    n_dislikes = min(int(likes * random.uniform(0.03, 0.15)), len(restantes))
    votantes_dislike = random.sample(restantes, n_dislikes)

    for uid in votantes_like:
        comment_vote_rows.append({
            "vote_id": comment_vote_id, "comment_id": row.comment_id,
            "user_id": uid, "vote_type": "like",
            "created_at": row.created_at + timedelta(hours=random.uniform(0, 800)),
        })
        comment_vote_id += 1
    for uid in votantes_dislike:
        comment_vote_rows.append({
            "vote_id": comment_vote_id, "comment_id": row.comment_id,
            "user_id": uid, "vote_type": "dislike",
            "created_at": row.created_at + timedelta(hours=random.uniform(0, 800)),
        })
        comment_vote_id += 1

df_comment_vote = pd.DataFrame(comment_vote_rows)
print("\ncomment_vote:", df_comment_vote.shape)
print(df_comment_vote.head())

# --- proposal_vote: engajamento agregado por proposta -----------------------
proposal_vote_rows = []
pv_id = 1
for prop in df_proposal.itertuples():
    n_votantes = random.randint(int(N_USERS * 0.15), int(N_USERS * 0.45))
    votantes = random.sample(all_user_ids, n_votantes)
    for uid in votantes:
        tipo = np.random.choice(["like", "dislike"], p=[0.74, 0.26])
        proposal_vote_rows.append({
            "vote_id": pv_id, "proposal_id": prop.proposal_id, "user_id": uid,
            "vote_type": tipo,
            "created_at": prop.created_at + timedelta(days=random.uniform(0, 300)),
        })
        pv_id += 1
df_proposal_vote = pd.DataFrame(proposal_vote_rows)
print("\nproposal_vote:", df_proposal_vote.shape)
print(df_proposal_vote.head())

# --- proposal_support ---------------------------------------------------
proposal_support_rows = []
ps_id = 1
for prop in df_proposal.itertuples():
    n_apoiadores = random.randint(int(N_USERS * 0.05), int(N_USERS * 0.20))
    for uid in random.sample(all_user_ids, n_apoiadores):
        proposal_support_rows.append({
            "support_id": ps_id, "proposal_id": prop.proposal_id, "user_id": uid,
            "created_at": prop.created_at + timedelta(days=random.uniform(0, 300)),
        })
        ps_id += 1
df_proposal_support = pd.DataFrame(proposal_support_rows)
print("\nproposal_support:", df_proposal_support.shape)
print(df_proposal_support.head())

# --- proposal_share -------------------------------------------------------
proposal_share_rows = []
sh_id = 1
for prop in df_proposal.itertuples():
    n_compart = random.randint(int(N_USERS * 0.03), int(N_USERS * 0.12))
    for uid in random.sample(all_user_ids, n_compart):
        proposal_share_rows.append({
            "share_id": sh_id, "proposal_id": prop.proposal_id, "user_id": uid,
            "shared_at": prop.created_at + timedelta(days=random.uniform(0, 300)),
        })
        sh_id += 1
df_proposal_share = pd.DataFrame(proposal_share_rows)
print("\nproposal_share:", df_proposal_share.shape)
print(df_proposal_share.head())


comment_vote: (2514, 5)
   vote_id  comment_id  user_id vote_type                 created_at
0        1          38      108      like 2026-02-23 09:57:24.998398
1        2          71      298      like 2026-05-24 09:16:11.647928
2        3          71      369      like 2026-06-07 13:56:10.943756
3        4          74       47      like 2026-01-07 10:40:24.496323
4        5          74       10      like 2025-12-22 00:23:57.373055

proposal_vote: (267, 5)
   vote_id  proposal_id  user_id vote_type                 created_at
0        1            1       14      like 2025-07-30 13:41:46.319743
1        2            1      294      like 2026-03-11 09:51:22.057095
2        3            1      114      like 2026-03-05 17:14:24.243121
3        4            1      160      like 2026-05-01 10:03:08.168300
4        5            1       62   dislike 2025-09-13 21:02:39.558057

proposal_support: (56, 4)
   support_id  proposal_id  user_id                 created_at
0           1            1

## Passo 5 — Validação e testes de fumaça (Data Quality)

Checagens executadas:

1. Ausência de nulos inesperados e conferência de `dtypes` em todas as 10 tabelas.
2. **Integridade referencial**: toda chave estrangeira precisa existir na tabela pai (14 relacionamentos checados).
3. **Unicidade de chaves primárias** em todas as tabelas.
4. Um resumo de negócio (engajamento por proposta) como sanity check final, exibindo `.head()`-style de cada DataFrame ao longo do processo.

In [9]:
print("\n" + "=" * 70)
print("SMOKE TESTS — Integridade referencial e tipos")
print("=" * 70)

tabelas = {
    "region": df_region, "state": df_state, "municipality": df_municipality,
    "app_user": df_app_user, "proposal": df_proposal, "comment": df_comment,
    "comment_vote": df_comment_vote, "proposal_vote": df_proposal_vote,
    "proposal_support": df_proposal_support, "proposal_share": df_proposal_share,
}
for nome, df in tabelas.items():
    print(f"\n--- {nome} ({len(df)} linhas) ---")
    print(df.dtypes)
    assert df.isnull().sum().sum() == 0, f"{nome} possui valores nulos inesperados"

fks = [
    ("state.region_id", df_state["region_id"], df_region["region_id"]),
    ("municipality.state_id", df_municipality["state_id"], df_state["state_id"]),
    ("app_user.municipality_id", df_app_user["municipality_id"], df_municipality["municipality_id"]),
    ("proposal.created_by", df_proposal["created_by"], df_app_user["user_id"]),
    ("comment.proposal_id", df_comment["proposal_id"], df_proposal["proposal_id"]),
    ("comment.user_id", df_comment["user_id"], df_app_user["user_id"]),
    ("comment_vote.comment_id", df_comment_vote["comment_id"], df_comment["comment_id"]),
    ("comment_vote.user_id", df_comment_vote["user_id"], df_app_user["user_id"]),
    ("proposal_vote.proposal_id", df_proposal_vote["proposal_id"], df_proposal["proposal_id"]),
    ("proposal_vote.user_id", df_proposal_vote["user_id"], df_app_user["user_id"]),
    ("proposal_support.proposal_id", df_proposal_support["proposal_id"], df_proposal["proposal_id"]),
    ("proposal_support.user_id", df_proposal_support["user_id"], df_app_user["user_id"]),
    ("proposal_share.proposal_id", df_proposal_share["proposal_id"], df_proposal["proposal_id"]),
    ("proposal_share.user_id", df_proposal_share["user_id"], df_app_user["user_id"]),
]
print("\n--- Checagem de chaves estrangeiras ---")
for nome_fk, serie_fk, serie_pk in fks:
    ok = serie_fk.isin(serie_pk).all()
    print(f"  [{'OK' if ok else 'FALHOU'}] {nome_fk} -> todos os valores existem na PK referenciada")
    assert ok, f"Integridade referencial quebrada em {nome_fk}"

print("\n--- Checagem de chaves primárias únicas ---")
pks = {
    "region_id": df_region, "state_id": df_state, "municipality_id": df_municipality,
    "user_id": df_app_user, "proposal_id": df_proposal, "comment_id": df_comment,
    "vote_id (comment_vote)": df_comment_vote.rename(columns={"vote_id": "vote_id (comment_vote)"}),
}
assert df_region["region_id"].is_unique
assert df_state["state_id"].is_unique
assert df_municipality["municipality_id"].is_unique
assert df_app_user["user_id"].is_unique
assert df_proposal["proposal_id"].is_unique
assert df_comment["comment_id"].is_unique
assert df_comment_vote["vote_id"].is_unique
assert df_proposal_vote["vote_id"].is_unique
assert df_proposal_support["support_id"].is_unique
assert df_proposal_share["share_id"].is_unique
print("  [OK] todas as chaves primárias são únicas")

print("\n--- Resumo de engajamento por proposta ---")
resumo = (
    df_proposal[["proposal_id", "title"]]
    .merge(df_comment.groupby("proposal_id").size().rename("n_comentarios"), on="proposal_id", how="left")
    .merge(df_proposal_vote[df_proposal_vote.vote_type == "like"].groupby("proposal_id").size().rename("n_likes"), on="proposal_id", how="left")
    .merge(df_proposal_support.groupby("proposal_id").size().rename("n_apoios"), on="proposal_id", how="left")
    .merge(df_proposal_share.groupby("proposal_id").size().rename("n_compartilhamentos"), on="proposal_id", how="left")
    .fillna(0)
)
print(resumo.to_string(index=False))


SMOKE TESTS — Integridade referencial e tipos

--- region (5 linhas) ---
region_id    int64
name           str
dtype: object

--- state (27 linhas) ---
state_id     int64
name           str
region_id    int64
dtype: object

--- municipality (44 linhas) ---
municipality_id    int64
name                 str
state_id           int64
dtype: object

--- app_user (600 linhas) ---
user_id                     int64
name                          str
email                         str
age                         int64
gender                        str
municipality_id             int64
created_at         datetime64[us]
dtype: object

--- proposal (1 linhas) ---
proposal_id             int64
title                     str
description               str
created_by              int64
created_at     datetime64[us]
dtype: object

--- comment (927 linhas) ---
comment_id              int64
proposal_id             int64
user_id                 int64
content                   str
created_at     datetime64[u

## Passo 6 — Análise de sentimentos com LLM (Ollama)

**O que esta etapa faz:** para cada comentário em `comment.content`, um modelo
de linguagem rodando localmente via [Ollama](https://ollama.com) classifica o
`sentimento` (Positivo / Neutro / Negativo), a `emoção` predominante e uma
`confiança` (0–100) da classificação. O resultado é salvo separadamente em
`output_csv/comment_sentimento.csv` (a tabela `comment.csv` original, sem
sentimento, continua sendo exportada normalmente no Passo de exportação final).

**Pré-requisitos para rodar esta célula com sucesso:**
1. Ter o [Ollama](https://ollama.com/download) instalado e em execução
   (`ollama serve`, ou o aplicativo desktop aberto).
2. Ter baixado o modelo configurado em `MODEL_NAME` (por padrão `gemma3:4b`):
   ```
   ollama pull gemma3:4b
   ```
3. Ter instalado as dependências Python da célula anterior (`ollama`, `tqdm`).

**Sobre o bug da versão anterior (718 de 727 comentários = `"Erro"`):**
A causa raiz era a ausência de uma checagem prévia de que o Ollama estava
acessível. Sem essa checagem, *toda* chamada `ollama.chat()` falhava
imediatamente (erro de conexão), o que ficava mascarado por três problemas
adicionais no código original:

| # | Problema | Efeito |
|---|----------|--------|
| 1 | Erros eram salvos no mesmo cache (`_cache`) usado para sucessos | Uma falha passageira em um comentário "carimbava" erro para sempre em qualquer outro comentário com texto idêntico, mesmo em reexecuções |
| 2 | A mensagem real do erro (`str(e)`) só era gravada dentro da coluna `emocao` do DataFrame | O usuário via apenas a contagem de `"Erro"`, sem nenhuma pista do motivo (Ollama fora do ar? Modelo não baixado? JSON inválido?) |
| 3 | `num_predict: 80` é pouco para o JSON de resposta em alguns modelos | Mesmo com o Ollama funcionando, a resposta podia ser cortada no meio e o `json.loads()` falhar |
| 4 | Nenhuma tentativa de retry | Qualquer falha de rede pontual descartava o comentário definitivamente |

**Correções aplicadas na célula a seguir:**
- `verificar_ambiente_ollama()` roda **antes** do loop principal: confirma que
  o servidor responde e que o modelo configurado está entre os modelos
  baixados. Se algo estiver faltando, a análise é **pulada** com uma mensagem
  de ação clara — em vez de gerar centenas de linhas de `"Erro"` silenciosas.
- Apenas classificações **bem-sucedidas** entram no cache.
- Até 2 novas tentativas (retry) por comentário em caso de falha.
- `num_predict` aumentado e parsing mais tolerante (remove blocos
  ` ```json ... ``` ` que alguns modelos adicionam mesmo com `format="json"`).
- O valor de `"sentimento"` retornado pelo modelo é validado contra o
  conjunto esperado (`Positivo`/`Neutro`/`Negativo`); qualquer outra coisa
  cai para `"Neutro"` em vez de propagar um valor inesperado para a base.
- Ao final, se ainda restarem erros após as tentativas, a célula imprime uma
  **amostra das mensagens de erro reais** (não apenas a contagem), para
  facilitar o diagnóstico.


In [10]:
%pip install ollama tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ==========================================================
# PASSO 6 - ANÁLISE DE SENTIMENTOS COM OLLAMA (VERSÃO CORRIGIDA)
# ==========================================================
# Ver célula de documentação acima para o histórico do bug original
# (718 de 727 comentários retornando "Erro") e o racional de cada
# correção aplicada aqui.

import json
import os
import re
import time

import ollama
import pandas as pd
from tqdm.auto import tqdm

MODEL_NAME = "gemma3:4b"      # ou qwen2.5:7b / mistral:7b
TEMPERATURE = 0
NUM_PREDICT = 200             # aumentado (era 80) para evitar JSON cortado
MAX_TENTATIVAS = 3            # 1 tentativa original + até 2 retries
SENTIMENTOS_VALIDOS = {"Positivo", "Neutro", "Negativo"}

_cache = {}          # só guarda classificações BEM-SUCEDIDAS (ver bug #1)
_erros_amostra = []  # guarda mensagens de erro reais para diagnóstico


def _extrair_nomes_modelos(resposta_list):
    """Lê a lista de modelos baixados localmente, suportando tanto o
    formato antigo do client `ollama` (dict com chave 'models') quanto o
    novo (objeto ListResponse com atributo .models) — a biblioteca mudou
    esse formato entre versões."""
    if isinstance(resposta_list, dict):
        modelos = resposta_list.get("models", [])
    else:
        modelos = getattr(resposta_list, "models", [])

    nomes = []
    for m in modelos or []:
        nome = m.get("model") if isinstance(m, dict) else getattr(m, "model", None)
        if nome:
            nomes.append(nome)
    return nomes


def verificar_ambiente_ollama(model_name):
    """Checagem prévia (preflight): confirma que o servidor Ollama está
    acessível e que `model_name` já foi baixado, ANTES de processar
    qualquer comentário. Levanta RuntimeError com uma mensagem de ação
    clara em caso de problema, em vez de deixar o erro estourar 727
    vezes dentro do loop principal."""
    try:
        resposta = ollama.list()
    except Exception as e:
        raise RuntimeError(
            "Não foi possível conectar ao servidor Ollama "
            "(http://localhost:11434).\n"
            "  -> Verifique se o Ollama está instalado e em execução:\n"
            "     inicie o aplicativo, ou rode `ollama serve` em um terminal.\n"
            f"  -> Detalhe técnico original: {type(e).__name__}: {e}"
        ) from e

    nomes_disponiveis = _extrair_nomes_modelos(resposta)
    if not any(model_name == n or n.startswith(model_name) for n in nomes_disponiveis):
        raise RuntimeError(
            f"O modelo '{model_name}' não está entre os modelos baixados "
            f"localmente ({nomes_disponiveis or 'nenhum modelo encontrado'}).\n"
            f"  -> Baixe-o antes de continuar: `ollama pull {model_name}`"
        )

    print(f"[OK] Ollama acessível e modelo '{model_name}' disponível.")


def _parsear_resposta_json(conteudo):
    """Faz o parsing tolerante da resposta do modelo: remove blocos
    ```json ... ``` que alguns modelos adicionam mesmo com format='json',
    e só então tenta json.loads()."""
    conteudo = conteudo.strip()
    conteudo = re.sub(r"^```(json)?", "", conteudo, flags=re.IGNORECASE).strip()
    conteudo = re.sub(r"```$", "", conteudo).strip()
    return json.loads(conteudo)


def analisar_sentimento(texto):
    if pd.isna(texto) or str(texto).strip() == "":
        return pd.Series(["Neutro", "Sem conteúdo", 0])

    texto = str(texto).strip()

    # Evita reprocessar comentários repetidos — mas só reaproveita
    # resultados que já foram BEM-SUCEDIDOS (correção do bug #1).
    if texto in _cache:
        return pd.Series(_cache[texto])

    prompt = f"""
Você é um especialista em análise de sentimentos.

Analise o comentário abaixo.

Responda SOMENTE com JSON válido.

{{
    "sentimento":"Positivo|Neutro|Negativo",
    "emocao":"Alegria|Frustração|Raiva|Tristeza|Esperança|Indiferença|Outro",
    "confianca":95
}}

Comentário:
{texto}
"""

    ultimo_erro = None
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            resposta = ollama.chat(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                format="json",
                options={"temperature": TEMPERATURE, "num_predict": NUM_PREDICT},
            )
            resultado = _parsear_resposta_json(resposta["message"]["content"])

            sentimento = resultado.get("sentimento", "Neutro")
            if sentimento not in SENTIMENTOS_VALIDOS:
                sentimento = "Neutro"  # valor fora do esperado vira Neutro, não propaga lixo

            emocao = resultado.get("emocao", "Outro")

            try:
                confianca = int(resultado.get("confianca", 80))
            except (TypeError, ValueError):
                confianca = 80

            dados = [sentimento, emocao, confianca]
            _cache[texto] = dados  # só cacheia sucesso
            return pd.Series(dados)

        except Exception as e:
            ultimo_erro = e
            if tentativa < MAX_TENTATIVAS:
                time.sleep(0.5 * tentativa)  # pequeno backoff antes do retry
            continue

    # Esgotadas as tentativas: NÃO cacheia o erro (correção do bug #1),
    # para permitir nova tentativa numa reexecução futura da célula.
    _erros_amostra.append(str(ultimo_erro))
    return pd.Series(["Erro", str(ultimo_erro), 0])


print("=" * 70)
print("ANÁLISE DE SENTIMENTOS COM LLM (OLLAMA)")
print("=" * 70)

try:
    verificar_ambiente_ollama(MODEL_NAME)
    ollama_disponivel = True
except RuntimeError as e:
    ollama_disponivel = False
    print("\n" + "!" * 70)
    print("ANÁLISE DE SENTIMENTOS PULADA")
    print("!" * 70)
    print(str(e))
    print(
        "\nA análise não foi executada para não gerar centenas de linhas de "
        "'Erro'. Corrija o problema acima e execute esta célula novamente."
    )

df_comment_full["sentimento"] = pd.NA
df_comment_full["emocao"] = pd.NA
df_comment_full["confianca"] = pd.NA

if ollama_disponivel:
    tqdm.pandas()
    df_comment_full[["sentimento", "emocao", "confianca"]] = (
        df_comment_full["content"].fillna("").progress_apply(analisar_sentimento)
    )

# DataFrame final (com ou sem sentimento, dependendo da disponibilidade do Ollama)
df_comment = df_comment_full.copy()

if ollama_disponivel:
    print("\nResumo da análise")
    print(df_comment["sentimento"].value_counts())

    validos = df_comment["sentimento"] != "Erro"
    print("\nConfiança média (apenas classificações válidas):")
    print(round(df_comment.loc[validos, "confianca"].mean(), 2) if validos.any() else "N/A")

    n_erros = (~validos).sum()
    print("\nComentários analisados:", len(df_comment))
    print(f"Comentários com erro após {MAX_TENTATIVAS} tentativas: {n_erros}")

    if n_erros:
        print(f"\n[ATENÇÃO] Amostra de mensagens de erro reais (até 5 de {len(_erros_amostra)}):")
        for msg in _erros_amostra[:5]:
            print("  -", msg)
    else:
        print("\n[OK] Nenhum erro registrado na análise de sentimentos.")

    os.makedirs("output_csv", exist_ok=True)
    df_comment.to_csv("output_csv/comment_sentimento.csv", index=False, encoding="utf-8-sig")
    print("\nArquivo salvo:")
    print("output_csv/comment_sentimento.csv")
else:
    print("\n[INFO] output_csv/comment_sentimento.csv NÃO foi gerado nesta execução.")

ANÁLISE DE SENTIMENTOS COM LLM (OLLAMA)
[OK] Ollama acessível e modelo 'gemma3:4b' disponível.


100%|██████████| 727/727 [1:19:39<00:00,  6.57s/it]


Resumo da análise
sentimento
Negativo    387
Positivo    193
Neutro      147
Name: count, dtype: int64

Confiança média (apenas classificações válidas):
89.23

Comentários analisados: 727
Comentários com erro após 3 tentativas: 0

[OK] Nenhum erro registrado na análise de sentimentos.

Arquivo salvo:
output_csv/comment_sentimento.csv


## Exportação — CSVs individuais por tabela

Gera um CSV por tabela em `output_csv/`, prontos para importar em qualquer banco relacional.

In [1]:
import os

os.makedirs("output_csv", exist_ok=True)
for nome, df in tabelas.items():
    caminho = f"output_csv/{nome}.csv"
    df.to_csv(caminho, index=False, encoding="utf-8-sig")
    print(f"  {caminho}  ({len(df)} linhas)")


NameError: name 'tabelas' is not defined